In [1]:
!pip install -q tensorflow scikit-learn seaborn opencv-python tqdm

^C



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os, zipfile, time, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras import layers, Model

In [ ]:
url = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"

os.makedirs("data", exist_ok=True)

if not os.path.exists("data/UCI_HAR"):
    urllib.request.urlretrieve(url, "data/har.zip")
    with zipfile.ZipFile("data/har.zip") as z:
        z.extractall("data")
    os.rename("data/UCI HAR Dataset", "data/UCI_HAR")

In [ ]:
root = "data/UCI_HAR"

signals = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]

def load_x(split):
    path = f"{root}/{split}/Inertial Signals"
    return np.stack([
        np.loadtxt(f"{path}/{s}_{split}.txt") for s in signals
    ], axis=-1)

def load_y(split):
    return np.loadtxt(f"{root}/{split}/y_{split}.txt", dtype=int) - 1

X = load_x("train")
y = load_y("train")

X_test = load_x("test")
y_test = load_y("test")

print(X.shape, y.shape)
print(X_test.shape, y_test.shape)

In [ ]:
rng = np.random.default_rng(42)

idx = np.concatenate([
    rng.choice(np.where(y == c)[0], 300, replace=False)
    for c in range(6)
])

rng.shuffle(idx)

X, y = X[idx], y[idx]

print(X.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=.15,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = (X_train - mean) / std
X_val   = (X_val - mean) / std
X_test  = (X_test - mean) / std

In [ ]:
names = [
    "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
    "SITTING", "STANDING", "LAYING"
]

print("Training :", X_train.shape)
print("Validation:", X_val.shape)
print("Testing :", X_test.shape)
print("Classes :", len(names))
print("Features:", X_train.shape[-1])
print("Sequence:", X_train.shape[1])

In [ ]:
for c in [0, 1, 3]:
    i = np.where(y_train == c)[0][0]

    plt.figure(figsize=(10, 3))
    for j in [0, 3, 6]:
        plt.plot(X_train[i, :, j], label=signals[j])

    plt.title(names[c])
    plt.xlabel("Time step")
    plt.ylabel("Sensor value")
    plt.legend()
    plt.show()

In [ ]:
Wx, Wh, b = .5, .8, .1
xs = [.5, .7, .2]
h = 0

for x in xs:
    h = np.tanh(Wx*x + Wh*h + b)
    print(h)

In [ ]:
def build(model_type, T=128):
    inp = layers.Input((T, 9))

    r = {
        "RNN": layers.SimpleRNN,
        "LSTM": layers.LSTM,
        "GRU": layers.GRU
    }[model_type]

    x = r(32)(inp)
    x = layers.Dropout(.2)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(6, activation="softmax")(x)

    model = Model(inp, out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
models, histories, times = {}, {}, {}

for name in ["RNN", "LSTM", "GRU"]:
    print("\n", name)

    model = build(name)

    start = time.time()

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=32,
        verbose=1
    )

    models[name] = model
    histories[name] = history
    times[name] = time.time() - start

In [ ]:
for name, h in histories.items():

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.plot(h.history["loss"], label="Train")
    plt.plot(h.history["val_loss"], label="Validation")
    plt.title(f"{name} Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(h.history["accuracy"], label="Train")
    plt.plot(h.history["val_accuracy"], label="Validation")
    plt.title(f"{name} Accuracy")
    plt.legend()

    plt.show()

In [ ]:
results = []

predictions = {}

for name, model in models.items():

    p = np.argmax(model.predict(X_test, verbose=0), axis=1)
    predictions[name] = p

    results.append([
        name,
        accuracy_score(y_test, p) * 100,
        precision_score(y_test, p, average="macro") * 100,
        recall_score(y_test, p, average="macro") * 100,
        f1_score(y_test, p, average="macro") * 100,
        model.count_params(),
        times[name]
    ])

results = pd.DataFrame(
    results,
    columns=[
        "Model", "Accuracy", "Precision", "Recall",
        "F1", "Parameters", "Training Time"
    ]
)

results

In [ ]:
for name, p in predictions.items():

    cm = confusion_matrix(y_test, p)

    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=names,
        yticklabels=names
    )

    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
results.set_index("Model")[["Accuracy", "F1"]].plot(
    kind="bar",
    figsize=(8, 5)
)

plt.ylabel("Score (%)")
plt.title("RNN vs LSTM vs GRU")
plt.xticks(rotation=0)
plt.show()

In [ ]:
seq_results = []

for T in [32, 64, 128]:

    for name in ["RNN", "LSTM", "GRU"]:

        model = build(name, T)

        model.fit(
            X_train[:, :T],
            y_train,
            validation_data=(X_val[:, :T], y_val),
            epochs=10,
            batch_size=32,
            verbose=0
        )

        p = np.argmax(
            model.predict(X_test[:, :T], verbose=0),
            axis=1
        )

        seq_results.append([
            T,
            name,
            f1_score(y_test, p, average="macro")
        ])

seq_results = pd.DataFrame(
    seq_results,
    columns=["Sequence Length", "Model", "F1"]
)

seq_results

In [ ]:
sns.lineplot(
    data=seq_results,
    x="Sequence Length",
    y="F1",
    hue="Model",
    marker="o"
)

plt.title("Sequence Length vs Test F1")
plt.show()

In [ ]:
import cv2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [ ]:
cnn = MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

cnn.trainable = False

print("CNN feature dimension:", cnn.output_shape[-1])

In [ ]:
def frames(video, n=10):

    cap = cv2.VideoCapture(video)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    ids = np.linspace(0, total - 1, n).astype(int)

    imgs = []

    for i in ids:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, frame = cap.read()

        if ok:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))
            imgs.append(frame)

    cap.release()

    return np.array(imgs)

In [ ]:
def video_features(video):

    x = frames(video)
    x = preprocess_input(x.astype("float32"))

    return cnn.predict(x, verbose=0)

In [ ]:
inp = layers.Input((10, 1280))

x = layers.LSTM(32)(inp)
x = layers.Dense(16, activation="relu")(x)
out = layers.Dense(3, activation="softmax")(x)

video_model = Model(inp, out)

video_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

video_model.summary()

In [ ]:
N = 5000
T = 4
V = 10

X_seq = np.random.randint(1, V, (N, T))
Y_seq = X_seq[:, ::-1]

print(X_seq[:5])
print(Y_seq[:5])

In [ ]:
inp = layers.Input((T,))
x = layers.Embedding(V, 16)(inp)

_, h, c = layers.LSTM(
    32,
    return_state=True
)(x)

dec = layers.RepeatVector(T)(h)

dec = layers.LSTM(
    32,
    return_sequences=True
)(dec, initial_state=[h, c])

out = layers.Dense(
    V,
    activation="softmax"
)(dec)

seq2seq = Model(inp, out)

seq2seq.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

seq2seq.summary()

In [ ]:
history = seq2seq.fit(
    X_seq,
    Y_seq[..., None],
    validation_split=.2,
    epochs=20,
    batch_size=64
)

In [ ]:
p = np.argmax(
    seq2seq.predict(X_seq[:5], verbose=0),
    axis=-1
)

for x, y, pred in zip(X_seq[:5], Y_seq[:5], p):
    print("Input :", x)
    print("Actual:", y)
    print("Pred  :", pred)
    print()

In [ ]:
pred = np.argmax(
    seq2seq.predict(X_seq, verbose=0),
    axis=-1
)

token_acc = np.mean(pred == Y_seq)
sequence_acc = np.mean(np.all(pred == Y_seq, axis=1))

print("Token accuracy   :", token_acc)
print("Sequence accuracy:", sequence_acc)